In [ ]:
# Visualize routing using the color of the employee
from matplotlib.lines import Line2D

# Copy of the route plot with adjusted pip colors:
# - employee pips are black
# - client pips use the color of their assigned employee
def _route_geometries_from_path_nodes_colored(graph, path_nodes, weight="travel_time_min"):
    route_geometries = []
    for node_u, node_v in pairwise(path_nodes):
        edge_bundle = graph.get_edge_data(node_u, node_v)
        if not edge_bundle:
            route_geometries.append(
                LineString(
                    [
                        (graph.nodes[node_u]["x"], graph.nodes[node_u]["y"]),
                        (graph.nodes[node_v]["x"], graph.nodes[node_v]["y"]),
                    ]
                )
            )
            continue

        best_key = min(
            edge_bundle,
            key=lambda candidate_key: edge_bundle[candidate_key].get(weight, float("inf")),
        )
        selected_edge = edge_bundle[best_key]
        geometry = selected_edge.get("geometry")
        if geometry is None or geometry.is_empty:
            geometry = LineString(
                [
                    (graph.nodes[node_u]["x"], graph.nodes[node_u]["y"]),
                    (graph.nodes[node_v]["x"], graph.nodes[node_v]["y"]),
                ]
            )
        route_geometries.append(geometry)

    return route_geometries

edges_plot = edges_gdf.to_crs(epsg=3857)
clients_plot = clients_gdf.to_crs(epsg=3857)
employee_points = employees_gdf.to_crs(epsg=3857)
plot_colors = plt.get_cmap("tab20", max(1, len(sequential_assignment["assignments"])))

fig, ax = plt.subplots(figsize=(13, 13))
edges_plot.plot(ax=ax, color="lightgray", linewidth=0.5, alpha=0.55)

legend_handles = []
client_color_map = {}

for idx, assignment in enumerate(sequential_assignment["assignments"]):
    color = plot_colors(idx)
    route_geometries = []

    for leg in assignment["fitness_result"]["legs"]:
        path_nodes = leg.get("path_nodes") or []
        if len(path_nodes) < 2:
            continue
        route_geometries.extend(
            _route_geometries_from_path_nodes_colored(
                road_graph,
                path_nodes,
                weight="travel_time_min",
            )
        )

    served_indices = assignment.get("served_client_indices", [])
    for client_idx in served_indices:
        client_color_map[int(client_idx)] = color

    if route_geometries:
        route_gdf = gpd.GeoDataFrame(geometry=route_geometries, crs="EPSG:4326").to_crs(epsg=3857)
        route_gdf.plot(ax=ax, color=color, linewidth=2.4, alpha=0.95)

    legend_handles.append(Line2D([0], [0], color=color, lw=2.4, label=assignment["employee_name"]))

# Plot each client's pip in the color of their assigned employee
for client_idx, color in client_color_map.items():
    if 0 <= client_idx < len(clients_plot):
        clients_plot.iloc[[client_idx]].plot(ax=ax, color=color, markersize=22, alpha=0.95)

# Optional: show unassigned clients in gray if any
unassigned_indices = [i for i in range(len(clients_plot)) if i not in client_color_map]
if unassigned_indices:
    clients_plot.iloc[unassigned_indices].plot(ax=ax, color="gray", markersize=18, alpha=0.65)

# Employee pips in black
employee_points.plot(ax=ax, color="black", markersize=28, alpha=0.95)

legend_handles.extend([
    Line2D([0], [0], marker="o", color="w", markerfacecolor="black", markersize=9, label="Employees"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="Unassigned clients"),
])

ax.set_title("Employee routes with client pips colored by assigned employee", fontsize=14)
ax.set_axis_off()
ax.legend(handles=legend_handles, loc="best", title="Employees")
plt.tight_layout()
plt.show()

In [ ]:
# Test cell: running tally of client allocation in the first 100 clients.
test_client_count = 100
route_order = list(initial_population[0])
test_route = route_order[: min(test_client_count, len(route_order))]

# Re-run assignment only for this test window.
sequential_assignment_test = assign_clients_to_employees_sequentially(
    heerlen_edge_table,
    clients_gdf,
    employees_gdf,
    test_route,
    weight_column="travel_time_min",
)

total_in_test = len(test_route)
if total_in_test == 0:
    raise ValueError("Test route is empty, cannot compute allocation tally.")

print(f"=== CLIENT ALLOCATION TALLY (first {total_in_test} clients) ===")
print("Format: employee_name | this employee count | cumulative served | cumulative %")

cumulative_served = 0
for assignment in sequential_assignment_test["assignments"]:
    employee_name = assignment.get("employee_name", assignment.get("employee_index"))
    served = assignment.get("served_client_indices", [])
    employee_count = len(served)
    cumulative_served += employee_count
    cumulative_pct = cumulative_served / total_in_test * 100.0

    print(
        f"{employee_name} | +{employee_count} | total={cumulative_served} | {cumulative_pct:.1f}%"
    )

remaining_total = len(sequential_assignment_test["remaining_client_indices"])
remaining_pct = remaining_total / total_in_test * 100.0

print("\n=== FINAL SUMMARY ===")
print(f"Assigned: {cumulative_served}/{total_in_test} ({(cumulative_served / total_in_test * 100.0):.1f}%)")
print(f"Unassigned: {remaining_total}/{total_in_test} ({remaining_pct:.1f}%)")

if cumulative_served > 0:
    print(f"Last served position (0-based): {cumulative_served - 1}")
    print(f"Last served position (1-based): {cumulative_served}")

if remaining_total > 0:
    print(f"Next unassigned position (0-based): {cumulative_served}")
    print(f"Next unassigned position (1-based): {cumulative_served + 1}")
    print("Remaining client indices from the test list:")
    print(sequential_assignment_test["remaining_client_indices"])

In [ ]:
# Improve results with two-parent crossover + diversity controls + lineage tracking.
# This version is faster to run (less console spam, early stop on stagnation) and explores more routes.

import json

EPOCH_IMPROVEMENT_COUNT_CROSSOVER = 150
ELITE_KEEP_CROSSOVER = 20
PARENT_POOL_SIZE = 50
CROSSOVER_CHILDREN_TARGET = POPULATION_SIZE
CROSSOVER_SWAP_MUTATION_RATE = 0.25
RANDOM_IMMIGRANTS_PER_EPOCH = 5
NO_IMPROVEMENT_PATIENCE = 25
PRINT_EVERY = 5

if 'population_evaluations' not in globals() or 'population_results_df' not in globals():
    raise ValueError('Run the population evaluation cell first so the initial ranking is available.')

def _to_builtin_lineage(value):
    if isinstance(value, dict):
        return {str(k): _to_builtin_lineage(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_builtin_lineage(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

def _route_repair(route_candidate):
    expected = set(range(len(route_candidate)))
    seen = set()
    duplicates_pos = []
    repaired = [int(x) for x in route_candidate]
    for pos, client_idx in enumerate(repaired):
        if client_idx in seen or client_idx not in expected:
            duplicates_pos.append(pos)
        else:
            seen.add(client_idx)
    missing = [idx for idx in range(len(repaired)) if idx not in seen]
    np.random.shuffle(missing)
    for pos in duplicates_pos:
        if not missing:
            break
        repaired[pos] = int(missing.pop())
    return repaired

def crossover_mutation_random(parent_one, parent_two):
    if len(parent_one) != len(parent_two):
        raise ValueError('Both parents must have the same length.')
    n = len(parent_one)
    if n <= 1:
        return [int(x) for x in parent_one]
    cut = int(np.random.randint(1, n))
    child = [int(x) for x in parent_one[:cut]] + [int(x) for x in parent_two[cut:]]
    return _route_repair(child)

lineage_id_counter = {'value': 0}

def _new_individual_crossover(
    route_candidate,
    root_line_id=None,
    parent_one_id=None,
    parent_two_id=None,
    origin='seed_elite',
):
    individual_id = f"xind_{lineage_id_counter['value']}"
    lineage_id_counter['value'] += 1
    if root_line_id is None:
        root_line_id = individual_id
    return {
        'individual_id': individual_id,
        'root_line_id': root_line_id,
        'parent_one_id': parent_one_id,
        'parent_two_id': parent_two_id,
        'origin': origin,
        'route_candidate': [int(idx) for idx in route_candidate],
    }

# Feasibility diagnostic: lower bound on unassigned clients if only care-hours are considered.
client_care = pd.to_numeric(clients_gdf.get('care_hours', 0.0), errors='coerce').fillna(0.0).to_numpy()
employee_available_hours = []
for employee_index in range(len(employees_gdf)):
    row = employees_gdf.iloc[employee_index]
    available_units = _employee_available_time_units(row, 'travel_time_min')
    employee_available_hours.append(float(available_units) / 60.0)
total_care_hours = float(client_care.sum())
total_available_hours = float(np.sum(employee_available_hours))
care_sorted = np.sort(client_care)
care_cumsum = np.cumsum(care_sorted)
max_clients_by_care_only = int(np.searchsorted(care_cumsum, total_available_hours, side='right'))
theoretical_min_unassigned = int(len(client_care) - max_clients_by_care_only)
print('Care-hours feasibility check:')
print(f'  total_client_care_hours={total_care_hours:.2f}')
print(f'  total_employee_available_hours={total_available_hours:.2f}')
print(f'  theoretical_min_unassigned_by_care_only={theoretical_min_unassigned}')

initial_routes_crossover = [
    initial_population[int(row.population_index)]
    for row in population_results_df.head(ELITE_KEEP_CROSSOVER).itertuples(index=False)
 ]
current_population_crossover = [
    _new_individual_crossover(route_candidate=route, origin='seed_elite')
    for route in initial_routes_crossover
]

current_history_crossover = []
all_epoch_records_crossover = []
current_best_result_crossover = None
current_best_population_index_crossover = None
current_best_unassigned_clients_crossover = None
current_best_total_travel_time_crossover = None
current_best_individual_crossover = None
stagnation_counter = 0

for epoch in range(1, EPOCH_IMPROVEMENT_COUNT_CROSSOVER + 1):
    epoch_results_crossover = []

    for population_idx, individual in enumerate(current_population_crossover):
        route_candidate = individual['route_candidate']
        assignment_result = assign_clients_to_employees_sequentially_fast(
            clients_gdf,
            employees_gdf,
            route_candidate,
            client_lookup_df=CLIENT_TRAVEL_LOOKUP_DF,
            node_distance_cache=NODE_DISTANCE_CACHE,
            weight_column='travel_time_min',
        )

        assigned_clients = len(route_candidate) - len(assignment_result['remaining_client_indices'])
        unassigned_clients = len(assignment_result['remaining_client_indices'])

        epoch_item = {
            'epoch': int(epoch),
            'population_index': int(population_idx),
            'individual_id': individual['individual_id'],
            'root_line_id': individual['root_line_id'],
            'parent_one_id': individual['parent_one_id'],
            'parent_two_id': individual['parent_two_id'],
            'origin': individual['origin'],
            'route_candidate': route_candidate,
            'total_time': float(assignment_result['total_time']),
            'total_travel_time': float(assignment_result['total_travel_time']),
            'total_care_time': float(assignment_result['total_care_time']),
            'assigned_clients': int(assigned_clients),
            'unassigned_clients': int(unassigned_clients),
            'assignment_result': assignment_result,
        }
        epoch_results_crossover.append(epoch_item)

        all_epoch_records_crossover.append({
            'epoch': int(epoch),
            'population_index': int(population_idx),
            'individual_id': individual['individual_id'],
            'root_line_id': individual['root_line_id'],
            'parent_one_id': individual['parent_one_id'],
            'parent_two_id': individual['parent_two_id'],
            'origin': individual['origin'],
            'total_time': float(assignment_result['total_time']),
            'total_travel_time': float(assignment_result['total_travel_time']),
            'total_care_time': float(assignment_result['total_care_time']),
            'assigned_clients': int(assigned_clients),
            'unassigned_clients': int(unassigned_clients),
            'route_candidate_json': json.dumps(_to_builtin_lineage(route_candidate)),
            'remaining_client_indices_json': json.dumps(_to_builtin_lineage(assignment_result.get('remaining_client_indices', []))),
            'assignment_result_json': json.dumps(_to_builtin_lineage(assignment_result)),
        })

    epoch_results_df_crossover = pd.DataFrame(epoch_results_crossover)
    epoch_results_sorted_df_crossover = epoch_results_df_crossover.sort_values(
        by=['unassigned_clients', 'total_travel_time'],
        ascending=[True, True],
    ).reset_index(drop=True)

    epoch_best_row = epoch_results_sorted_df_crossover.iloc[0]
    epoch_best_item = epoch_results_crossover[int(epoch_best_row['population_index'])]

    improved = False
    if current_best_result_crossover is None or (
        int(epoch_best_row['unassigned_clients']) < current_best_unassigned_clients_crossover or
        (
            int(epoch_best_row['unassigned_clients']) == current_best_unassigned_clients_crossover and
            float(epoch_best_row['total_travel_time']) < current_best_total_travel_time_crossover
        )
    ):
        improved = True
        current_best_result_crossover = epoch_best_item['assignment_result']
        current_best_population_index_crossover = int(epoch_best_row['population_index'])
        current_best_unassigned_clients_crossover = int(epoch_best_row['unassigned_clients'])
        current_best_total_travel_time_crossover = float(epoch_best_row['total_travel_time'])
        current_best_individual_crossover = {
            'individual_id': epoch_best_item['individual_id'],
            'root_line_id': epoch_best_item['root_line_id'],
            'parent_one_id': epoch_best_item['parent_one_id'],
            'parent_two_id': epoch_best_item['parent_two_id'],
            'origin': epoch_best_item['origin'],
            'route_candidate': [int(idx) for idx in epoch_best_item['route_candidate']],
        }

    stagnation_counter = 0 if improved else stagnation_counter + 1

    current_history_crossover.append({
        'epoch': int(epoch),
        'best_total_time': float(current_best_result_crossover['total_time']),
        'best_total_travel_time': float(current_best_result_crossover['total_travel_time']),
        'best_total_care_time': float(current_best_result_crossover['total_care_time']),
        'best_assigned_clients': int(len(initial_population[0]) - len(current_best_result_crossover['remaining_client_indices'])),
        'best_unassigned_clients': int(len(current_best_result_crossover['remaining_client_indices'])),
        'best_individual_id': current_best_individual_crossover['individual_id'] if current_best_individual_crossover is not None else None,
        'best_root_line_id': current_best_individual_crossover['root_line_id'] if current_best_individual_crossover is not None else None,
        'stagnation_counter': int(stagnation_counter),
    })

    if epoch == 1 or epoch % PRINT_EVERY == 0 or improved:
        print(
            f"Epoch {epoch}/{EPOCH_IMPROVEMENT_COUNT_CROSSOVER}: "
            f"best unassigned={len(current_best_result_crossover['remaining_client_indices'])}, "
            f"best travel={current_best_result_crossover['total_travel_time']:.2f}, "
            f"stagnation={stagnation_counter}"
        )

    if stagnation_counter >= NO_IMPROVEMENT_PATIENCE:
        print(f"Early stop: no improvement for {NO_IMPROVEMENT_PATIENCE} epochs.")
        break

    parent_pool_count = min(PARENT_POOL_SIZE, len(epoch_results_sorted_df_crossover))
    parent_pool_indices = [
        int(idx) for idx in epoch_results_sorted_df_crossover.head(parent_pool_count)['population_index'].tolist()
    ]
    parent_pool = [epoch_results_crossover[idx] for idx in parent_pool_indices]

    elite_indices = [
        int(idx) for idx in epoch_results_sorted_df_crossover.head(ELITE_KEEP_CROSSOVER)['population_index'].tolist()
    ]
    elites = [epoch_results_crossover[idx] for idx in elite_indices]

    next_population_crossover = []
    if current_best_individual_crossover is not None:
        next_population_crossover.append(
            _new_individual_crossover(
                route_candidate=current_best_individual_crossover['route_candidate'],
                root_line_id=current_best_individual_crossover['root_line_id'],
                parent_one_id=current_best_individual_crossover['individual_id'],
                parent_two_id=None,
                origin='best_carryover',
            )
        )

    for elite_item in elites:
        next_population_crossover.append(
            _new_individual_crossover(
                route_candidate=elite_item['route_candidate'],
                root_line_id=elite_item['root_line_id'],
                parent_one_id=elite_item['individual_id'],
                parent_two_id=None,
                origin='elite_copy',
            )
        )

    immigrant_count = min(RANDOM_IMMIGRANTS_PER_EPOCH, max(0, CROSSOVER_CHILDREN_TARGET - len(next_population_crossover)))
    for _ in range(immigrant_count):
        random_route = np.random.permutation(len(initial_population[0])).tolist()
        next_population_crossover.append(
            _new_individual_crossover(route_candidate=random_route, origin='random_immigrant')
        )

    while len(next_population_crossover) < CROSSOVER_CHILDREN_TARGET:
        parent_positions = np.random.choice(len(parent_pool), size=2, replace=False)
        parent_one = parent_pool[int(parent_positions[0])]
        parent_two = parent_pool[int(parent_positions[1])]
        child_route = crossover_mutation_random(parent_one['route_candidate'], parent_two['route_candidate'])
        if np.random.rand() < CROSSOVER_SWAP_MUTATION_RATE:
            child_route = swap_mutation(child_route)
            child_origin = 'two_parent_crossover_plus_swap'
        else:
            child_origin = 'two_parent_crossover'
        next_population_crossover.append(
            _new_individual_crossover(
                route_candidate=child_route,
                root_line_id=parent_one['root_line_id'],
                parent_one_id=parent_one['individual_id'],
                parent_two_id=parent_two['individual_id'],
                origin=child_origin,
            )
        )

    current_population_crossover = next_population_crossover[:POPULATION_SIZE]

improvement_history_df_crossover = pd.DataFrame(current_history_crossover)
all_epoch_individuals_df_crossover = pd.DataFrame(all_epoch_records_crossover)
final_improvement_results_df_crossover = epoch_results_sorted_df_crossover.drop(columns=['route_candidate', 'assignment_result']).copy()

print('\nBest result after crossover improvement loop:')
print(f"population_index={current_best_population_index_crossover}")
print(f"total_time={current_best_result_crossover['total_time']:.2f}")
print(f"total_travel_time={current_best_result_crossover['total_travel_time']:.2f}")
print(f"total_care_time={current_best_result_crossover['total_care_time']:.2f}")
print(f"served_clients={len(initial_population[0]) - len(current_best_result_crossover['remaining_client_indices'])}")
print(f"remaining_clients={len(current_best_result_crossover['remaining_client_indices'])}")
if current_best_individual_crossover is not None:
    print(f"best_individual_id={current_best_individual_crossover['individual_id']}")
    print(f"best_root_line_id={current_best_individual_crossover['root_line_id']}")

output_dir = Path('../output/genetic_algorithm')
output_dir.mkdir(parents=True, exist_ok=True)
run_stamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

history_timestamped_path = output_dir / f'ga_crossover_history_{run_stamp}.csv'
history_latest_path = output_dir / 'ga_crossover_history_latest.csv'
final_timestamped_path = output_dir / f'ga_crossover_final_results_{run_stamp}.csv'
final_latest_path = output_dir / 'ga_crossover_final_results_latest.csv'
lineage_timestamped_path = output_dir / f'ga_crossover_all_individuals_{run_stamp}.csv'
lineage_latest_path = output_dir / 'ga_crossover_all_individuals_latest.csv'
best_timestamped_path = output_dir / f'ga_crossover_best_result_{run_stamp}.json'
best_latest_path = output_dir / 'ga_crossover_best_result_latest.json'

improvement_history_df_crossover.to_csv(history_timestamped_path, index=False)
improvement_history_df_crossover.to_csv(history_latest_path, index=False)
final_improvement_results_df_crossover.to_csv(final_timestamped_path, index=False)
final_improvement_results_df_crossover.to_csv(final_latest_path, index=False)
all_epoch_individuals_df_crossover.to_csv(lineage_timestamped_path, index=False)
all_epoch_individuals_df_crossover.to_csv(lineage_latest_path, index=False)

best_result_payload_crossover = {
    'run_stamp': run_stamp,
    'epochs': int(len(improvement_history_df_crossover)),
    'elite_keep': int(ELITE_KEEP_CROSSOVER),
    'parent_pool_size': int(PARENT_POOL_SIZE),
    'crossover_swap_mutation_rate': float(CROSSOVER_SWAP_MUTATION_RATE),
    'random_immigrants_per_epoch': int(RANDOM_IMMIGRANTS_PER_EPOCH),
    'population_index': int(current_best_population_index_crossover) if current_best_population_index_crossover is not None else None,
    'total_time': float(current_best_result_crossover['total_time']) if current_best_result_crossover is not None else None,
    'total_travel_time': float(current_best_result_crossover['total_travel_time']) if current_best_result_crossover is not None else None,
    'total_care_time': float(current_best_result_crossover['total_care_time']) if current_best_result_crossover is not None else None,
    'assigned_clients': int(len(initial_population[0]) - len(current_best_result_crossover['remaining_client_indices'])) if current_best_result_crossover is not None else None,
    'unassigned_clients': int(len(current_best_result_crossover['remaining_client_indices'])) if current_best_result_crossover is not None else None,
    'remaining_client_indices': [int(idx) for idx in current_best_result_crossover['remaining_client_indices']] if current_best_result_crossover is not None else [],
    'best_individual_id': current_best_individual_crossover['individual_id'] if current_best_individual_crossover is not None else None,
    'best_root_line_id': current_best_individual_crossover['root_line_id'] if current_best_individual_crossover is not None else None,
    'best_parent_one_id': current_best_individual_crossover['parent_one_id'] if current_best_individual_crossover is not None else None,
    'best_parent_two_id': current_best_individual_crossover['parent_two_id'] if current_best_individual_crossover is not None else None,
    'best_origin': current_best_individual_crossover['origin'] if current_best_individual_crossover is not None else None,
    'best_route_candidate': [int(idx) for idx in current_best_individual_crossover['route_candidate']] if current_best_individual_crossover is not None else [],
    'theoretical_min_unassigned_by_care_only': int(theoretical_min_unassigned),
}

with open(best_timestamped_path, 'w', encoding='utf-8') as f:
    json.dump(best_result_payload_crossover, f, indent=2)
with open(best_latest_path, 'w', encoding='utf-8') as f:
    json.dump(best_result_payload_crossover, f, indent=2)

print('\nSaved crossover results:')
print(f'- {history_timestamped_path}')
print(f'- {final_timestamped_path}')
print(f'- {lineage_timestamped_path}')
print(f'- {best_timestamped_path}')
print(f'- {history_latest_path}')
print(f'- {final_latest_path}')
print(f'- {lineage_latest_path}')
print(f'- {best_latest_path}')

print('\nEpoch summary (crossover):')
improvement_history_df_crossover